# 03 — Understanding the ground truth and the score (Phase 3)

**Goal:** see with our own eyes what a *true match* looks like, and understand exactly how we are scored.

1. How the competition score (macro F0.5) works, with worked examples.
2. Real matches side by side: easy ones, hard ones, many-match ones, and singletons (no match).
3. **Measure** what evidence true matches share (same name? same house number? same words?).
4. **Traps**: records that look like a match (same name) but are not.

In [1]:
import gc

import pandas as pd

pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 220)

TRAIN_DIR = "../dataset/train/"

---
## 1. How we are scored — macro F0.5 (from the official problem statement)

For **each** Source 1 entity separately:

- **Precision** = of the IDs we predicted, what fraction are correct?
- **Recall** = of the true IDs, what fraction did we find?
- **F0.5** = `1.25 × P × R / (0.25 × P + R)` — a blend of both that counts precision **twice as much** as recall.

Then the final score is the **average over all Source 1 entities** ("macro average").

Special cases (singletons = S1 entities with no true match):

| True matches | We predict | Score for this entity |
|---|---|---|
| none | nothing | **1.0** (correct "no match") |
| none | anything | **0.0** |
| some | nothing | **0.0** (recall = 0) |
| some | some | F0.5 formula |

In [2]:
def entity_f05(true_ids, predicted_ids):
    true_ids, predicted_ids = set(true_ids), set(predicted_ids)
    if not true_ids and not predicted_ids:
        return 1.0
    correct = len(true_ids & predicted_ids)
    if correct == 0:
        return 0.0
    precision = correct / len(predicted_ids)
    recall = correct / len(true_ids)
    return 1.25 * precision * recall / (0.25 * precision + recall)


# the example from the problem statement: expected 0.714
print(round(entity_f05({"S2-00047", "S3-00812"}, {"S2-00047", "S2-00193", "S3-00812"}), 3))

0.714


### What does "precision counts double" mean in practice?

Imagine an S1 entity with **4** true matches `{A, B, C, D}`. Compare different predictions:

In [3]:
truth = {"A", "B", "C", "D"}
scenarios = {
    "all 4 correct":                 {"A", "B", "C", "D"},
    "all 4 correct + 1 wrong":       {"A", "B", "C", "D", "X"},
    "3 correct":                     {"A", "B", "C"},
    "2 correct":                     {"A", "B"},
    "1 correct":                     {"A"},
    "2 correct + 2 wrong":           {"A", "B", "X", "Y"},
    "1 correct + 3 wrong":           {"A", "X", "Y", "Z"},
    "nothing":                       set(),
}
pd.DataFrame(
    [(name, len(pred & truth), len(pred - truth), round(entity_f05(truth, pred), 3)) for name, pred in scenarios.items()],
    columns=["prediction", "correct", "wrong", "F0.5"],
)

,prediction,correct,wrong,F0.5
0,all 4 correct,4,0,1.000
1,all 4 correct + 1 wrong,4,1,0.833
2,3 correct,3,0,0.938
3,2 correct,2,0,0.833
4,1 correct,1,0,0.625
5,2 correct + 2 wrong,2,2,0.500
6,1 correct + 3 wrong,1,3,0.250
7,nothing,0,0,0.000


**Takeaways**
- Finding only **2 of 4** correctly (no mistakes) scores 0.833 — better than finding all 4 but adding 2 wrong ones (0.625 for 2+2).
- One wrong ID costs less when we also found many correct ones.
- For a **singleton**, a single wrong ID drops the score from 1.0 to 0.0.

### Two "dumb" baselines on the full training ground truth

Any real model must beat these.

In [4]:
gt = pd.read_csv(TRAIN_DIR + "train_ground_truth.tsv", sep="\t", keep_default_na=False)
gt["true_ids"] = gt["matched_entity_ids"].str.split(",").where(gt["matched_entity_ids"] != "", None)
gt["n_true"] = gt["true_ids"].str.len().fillna(0).astype(int)

singleton_share = (gt["n_true"] == 0).mean()
print(f"Predict NOTHING for everyone        -> score = {singleton_share:.4f}  (only singletons score 1)")
print(f"Perfect answers for everyone        -> score = 1.0000")

Predict NOTHING for everyone        -> score = 0.0558  (only singletons score 1)
Perfect answers for everyone        -> score = 1.0000


So "predict nothing" scores only **0.056**. Since 94% of entities do have matches, we must find matches —
but carefully.

---
## 2. Real matches side by side

To stay within memory we take a **random sample of 20,000 Source 1 entities** and fetch only their matches
from Source 2 and Source 3.

In [5]:
sample_gt = gt.sample(20_000, random_state=1).reset_index(drop=True)

pairs = sample_gt.dropna(subset=["true_ids"]).explode("true_ids")[["source1_entity_id", "true_ids"]]
pairs = pairs.rename(columns={"true_ids": "match_id"}).reset_index(drop=True)
print("sampled S1 entities:", len(sample_gt), "| their true pairs:", len(pairs))

s1 = pd.read_csv(TRAIN_DIR + "train_source1.tsv", sep="\t", keep_default_na=False)
s1 = s1[s1["entity_id"].isin(sample_gt["source1_entity_id"])].reset_index(drop=True)
gc.collect()

matches = []
for source_number in [2, 3]:
    df = pd.read_csv(TRAIN_DIR + f"train_source{source_number}.tsv", sep="\t", keep_default_na=False)
    matches.append(df[df["entity_id"].isin(pairs["match_id"])])
    del df
    gc.collect()
matches = pd.concat(matches, ignore_index=True)
print("S1 rows:", len(s1), "| matched S2/S3 rows:", len(matches))

sampled S1 entities: 20000 | their true pairs: 69504


S1 rows: 20000 | matched S2/S3 rows: 69504


Now one wide table: each row is **one true pair** (S1 record + its matching record).
`merge` joins tables on a shared ID, like VLOOKUP in Excel.

In [6]:
true_pairs = (
    pairs
    .merge(s1.add_prefix("s1_"), left_on="source1_entity_id", right_on="s1_entity_id")
    .merge(matches.add_prefix("m_"), left_on="match_id", right_on="m_entity_id")
    .drop(columns=["s1_entity_id", "m_entity_id", "m_country"])
)
true_pairs["m_source"] = true_pairs["match_id"].str[:2]
print(true_pairs.shape)
true_pairs.head()

(69504, 8)


,source1_entity_id,match_id,s1_business_name,s1_business_address,s1_country,m_business_name,m_business_address,m_source
0,S1-41288513,S2-792601653,A Strategic Environmental,"2 Muldowney Circle, Unit Apartment B, Poughkeepsie, NY",US,A SOBTRNATMGIC ENVIRONMENTAL,"2 MULDOWNEY CIR, POUGHKEEPSIE, NY",S2
1,S1-41288513,S3-599697910,A Strategic Environmental,"2 Muldowney Circle, Unit Apartment B, Poughkeepsie, NY",US,A Strategic [Environmental],"2 Mulsowney Cir, # Apartment B, Poughkeepsie, New York",S3
2,S1-41288513,S3-290998331,A Strategic Environmental,"2 Muldowney Circle, Unit Apartment B, Poughkeepsie, NY",US,A Environmental (Strategic),"2 Muldowney Cir, # Apartment B, Poughkeepsie, New York",S3
3,S1-41288513,S3-460626156,A Strategic Environmental,"2 Muldowney Circle, Unit Apartment B, Poughkeepsie, NY",US,The Á Strategic Environmental,"2 Muldowney Cir, # Apartment B, Poughkeepsie, New York",S3
4,S1-256559799,S2-483613492,Lucknow Apparels Group,"4/792, Sector 4, Vikas Nagar, Lucknow, Uttar Pradesh",India,Lucknow Appare1s,"4/792., SECTOR 4, VIKAS NAGAR, LUCKNOW, उत्तर प्रदेश",S2


In [7]:
def show_entity(s1_id):
    s1_row = s1[s1["entity_id"] == s1_id]
    ids = pairs.loc[pairs["source1_entity_id"] == s1_id, "match_id"]
    print(f"\n########## {s1_id}  ->  {len(ids)} true match(es)")
    display(pd.concat([s1_row, matches[matches["entity_id"].isin(ids)]]).reset_index(drop=True))

### 2a. Easy matches — the name is the same after lowercasing

In [8]:
same_name = true_pairs["s1_business_name"].str.lower() == true_pairs["m_business_name"].str.lower()
for s1_id in true_pairs.loc[same_name, "source1_entity_id"].drop_duplicates().sample(2, random_state=0):
    show_entity(s1_id)


########## S1-508759191  ->  3 true match(es)


,entity_id,business_name,business_address,country
0,S1-508759191,Greenline Foundation,"544C/88/95A, Almas City Hardoi Road, Balaganj, Lucknow, Uttar Pradesh",India
1,S2-95492628,Greenline Fóundation,"BLOCK F-971 544C/88/95A, ALMAS CITY HARDOI ROAD, BALAGANJ, LUCKNOW...",India
2,S2-142948023,Greenline Foundation,"BLOCK F-971 544C/88/95A, ALMAS CITY HARDOI ROAD, BALAGANJ, LUCKNOW...",India
3,S3-790572878,Nexpyra,"544C/88/95A, Almas City Hardoi Road, Balaganj, Lucknow, Uttar Pradesh",India



########## S1-465639422  ->  3 true match(es)


,entity_id,business_name,business_address,country
0,S1-465639422,Orient Ventures Private Limited,"Plot.No.159, H.No.8-2-684/4137, 1St Floor Road.No.12, Banjara Hill...",India
1,S2-414548947,Orient Ventures Limited-Partners,"H.NO 59 , H.NO.8-2-684/4137, 1ST FLOOR ROAD.NO.12, BANJARA HILLS, ...",India
2,S2-886295649,ORIENT LIMITED PRIVATE (VENTURES),"NO 59 , H.NO.8-2-684/4137, 1ST FLOOR ROAD.NO.12, BANJARA HILLS, Te...",India
3,S3-927393720,ORIENT VENTURES PRIVATE LIMITED,"Plot.no.159., H.no.8-2-684/4137, 1St Floor Road.no.12, Banjara Hil...",India


### 2b. Entities with many matches (8 or more)

In [9]:
for s1_id in sample_gt.loc[sample_gt["n_true"] >= 8, "source1_entity_id"].head(2):
    show_entity(s1_id)


########## S1-681456683  ->  8 true match(es)


,entity_id,business_name,business_address,country
0,S1-681456683,American Program,"6005 Kary Lynn Drive, Fort Worth, TX",US
1,S2-678628335,American Prógram,"6005 KARY LYNN DR, FORT WORTH, TX",US
2,S2-454005529,AMERICAN PROGRAM,"6005 KARY LYNN DR, FORT WORTH, TX",US
3,S2-999263688,AMERlCAN PROGRAM (CO),"6005 KARY LYNN DRIEV, FORT WORTH, TX",US
4,S2-307177659,AMERICAN PROGRAM,,US
5,S2-144751089,American Program Inc.,"KARY LYNN DRIVE, FORT WORTH, TX",US
6,S3-806537495,American Program,"6005 Kary Lynn Drive, FT. Woth, Texas",US
7,S3-376623615,Zephvantagexylo Sys née American Program,"6005 Kary Lynn Drive, Fort Worth, TX",US
8,S3-536528085,American Program Inc.,,US



########## S1-117479787  ->  8 true match(es)


,entity_id,business_name,business_address,country
0,S1-117479787,Suresh Broadcasters Private Limited,"C/Osanjay Singh, H.No-176, Ward-7 Porsa Gandhi Nagar, Porsa, Moren...",India
1,S2-313212849,Private Suresh Broadcasters Limited,"C/OSANJAY SINGH, H.NO-176, WARD-7 PORSA GANDHI NAGAR, PORSA, Madhy...",India
2,S2-158629049,SURESHBROADCASTERS.COM,"78 C/OSANJAY SINGH, H.NO-176, WARD-7 PORSA GANDHI NAGAR, PORSA, मध...",India
3,S2-6188389,SBP,"H.NO 78 C/OSANJAY SINGH, PORSA, MORENA, Madhya Pradesh",India
4,S3-127786625,Suresh Broadcasters Private Ltd,"H.no 883 C/osanjay Singh, H.no-176, Ward-7 Porsa Gandhi Nagar, Mor...",India
5,S3-520999770,Suresh Broadcasters Private,"883 C/osanjay Singh, H.no-176, Ward-7 Porsa Gandhi Nagar, Morena, ...",India
6,S3-299052164,Suresh Broadcasters Private,,India
7,S3-816597875,Pyrabelo née Suresh Broadcasters Private Limited,"C/Osanjay Singh, H.No-176, Ward-7 Porsa Gandhi Nagar, Porsa, Moren...",India
8,S3-641943161,Suresh Broadcasters Pbvtae Limited,"Door No 883 C/osanjay Singh, H.no-176, Ward-7 Porsa Gandhi Nagar, ...",India


**Things to notice in these examples**
- **"née" = trade name / former name:** `Zephvantagexylo Sys née American Program`. The real name comes *after* `née`.
- **Completely different names** (`Nexpyra`, `SBP`) are still true matches — linked only by the address.
- **Extra numbers get injected** into S2/S3 addresses (`BLOCK F-971 544C/88/95A`, `H.no 883 C/osanjay Singh`),
  so "shares a number" is good evidence but "has a different number" is **not** proof of a non-match.
- Noise stacks up: typo + reordering + brackets + abbreviation in the same record (`ORIENT LIMITED PRIVATE (VENTURES)`).

### 2c. Hard matches — the names share **no word at all**

We need a tiny cleaning step to compare words: lowercase, replace symbols with spaces, split.
(`\p{L}` letter, `\p{N}` digit, `\p{M}` mark — see the regex trap in notebook 02.)
This is a *temporary* helper; proper cleaning is Phase 6.

In [10]:
def simple_words(text_series):
    return text_series.str.lower().str.replace(r"[^\p{L}\p{N}\p{M}\s]", " ", regex=True).str.split()

true_pairs["s1_name_words"] = simple_words(true_pairs["s1_business_name"])
true_pairs["m_name_words"] = simple_words(true_pairs["m_business_name"])
true_pairs["s1_addr_words"] = simple_words(true_pairs["s1_business_address"])
true_pairs["m_addr_words"] = simple_words(true_pairs["m_business_address"])

# row by row: do the two word lists have at least one word in common?
true_pairs["name_shares_word"] = [
    len(set(a) & set(b)) > 0 for a, b in zip(true_pairs["s1_name_words"], true_pairs["m_name_words"])
]

hard = true_pairs.loc[~true_pairs["name_shares_word"],
                      ["s1_business_name", "m_business_name", "s1_business_address", "m_business_address"]]
print(f"true pairs whose names share no word: {len(hard):,} of {len(true_pairs):,}")
hard.sample(12, random_state=0)

true pairs whose names share no word: 9,986 of 69,504


,s1_business_name,m_business_name,s1_business_address,m_business_address
2340,Pipefitters Local 426,#pipefitterslocal,"117 Canal Drive, Rockford, MN","ROCKFORD TOWNSHIP, MN, 117 CANAL DR"
38138,New Infra Private Limited,न्यू इंफ्रा प्राइवेट लिमिटेड,"Uttar Pradesh, Lucknow, B- 109 Vibhuti Khand Gomti Nagar","B- 109 VIBHUTI KHAND GOMTI NAGAR, LUCKNOW, उत्तर प्रदेश"
40913,Mchenry & Owens LLC,owensmchenry.com,"1250 Irving Avenue, City Of Oshkosh, WI","1250 IRVING AVE, CITY OF OSHKOSH, WI"
60889,Sun Constructions Private Limited,सन कंस्ट्रक्शंस प्राइवेट लिमिटेड,"61-62, Transport Centrepunjabi Bagh, New Delhi, West Delhi, Delhi","PLOT B3/403 61-62, WEST DELHI, NEW DELHI, दिल्ली"
35244,Green Engineering Private Limited,ഗ്രീൻ എൻജിനീയറിംഗ് പ്രൈവറ്റ് ലിമിറ്റഡ്,"Ap Vi/683, Josephine Villa Anjuthengu.P.O., Thiruvananthapuram, Tr...","THIRUVANANTHAPURAM, കേരളം, TRIVANDRUM, BLOCK B-301 AP VI/683"
48267,Ridge League LLC,Ariafaye,"337 Roberts Avenue, Norton City, VA","337 Roberts Ave, Norton City, Virginia"
19305,Star Baba Foods,स्टार बाबा फूड्स,"Maharashtra, Pune, Sr. No. 87, Veerbhadra Nagar, First Floor","MH, No ##568 First Floor, Pune"
3577,Vijay Hospitality Pvt Ltd,বিজয় হসপিটালিটি প্রাইভেট লিমিটেড,"Kolkata, 123 Chittaranjan Avenuep.S.Hare Street, West Bengal, Calc...","123 CHITTARANJAN AVENUEP.S.HARE STREET, KOLKATA, CALCUTTA, West Be..."
14773,Tech Alpha Business Private Limited,टेक अल्फा बिजनेस प्राइवेट लिमिटेड,"Ward No.2 Dnyaneshwar Nagar Wardha, Wardha, Maharashtra","AMRAVATI, WARD NO.D/2 DNYANESHWAR NAGAR WARDHA, महाराष्ट्र, WARDHA"
24465,First Infra Limited,ফার্স্ট ইনফ্রা লিমিটেড,"Chopra, Uttar Dinajpur, Chopra, North Dinajpur, West Bengal","Chopra, North Dinajpur, পশ্চিমবঙ্গ"


### 2d. Singletons — S1 entities with no match

In [11]:
s1[s1["entity_id"].isin(sample_gt.loc[sample_gt["n_true"] == 0, "source1_entity_id"])].head(8)

,entity_id,business_name,business_address,country
16,S1-625041866,Super 44 Auto Body Clinic,"387 Marguerite Avenue, South Floral Park, NY",US
27,S1-920297646,West Federal Oyster LLC,"301 Neals Lane, Gallatin, TN",US
54,S1-498399790,Luckey Vista Inc,"95164 Johnson Island, Tower, MN",US
59,S1-112704648,"N & I Teucrium, LLC","Paintsville, KY, 1119 Boyd Street",US
81,S1-152581295,Alpha Pharmaceuticals Private Limited,"Delhi, South Delhi, E-206Second Floor East Of Kailash, New Delhi",India
84,S1-708682519,Foundation Inspire Microfinance Clinic,"295, Bldg 74, Gala 1, 2 Indian Corporation, Mouje Gundavli, Taluk,...",India
109,S1-521773930,Goldman Smart Bleichroeder Fairfax County,"864 Sconset Lane, Fairfax County, VA",US
137,S1-751991890,Premier Solutions Private Limited,"806 Indraprasth Corporate, Opp. Venus Atlantis, B/S Safal Pagusus,...",India


A singleton **looks exactly like any other S1 record**. Nothing in the record itself says "I have no match";
we can only conclude it when no candidate is convincing enough. That is why the decision threshold (Phase 10)
matters so much.

---
## 3. Measuring the evidence in true pairs

For every true pair we compute simple yes/no checks, then the percentage of pairs where each is True.
This tells us **which signals are reliable** before we build any features.

| Check | Meaning |
|---|---|
| `name_exact` | names identical, character for character |
| `name_same_lower` | identical after lowercasing |
| `name_same_words` | same set of words (ignores order, case, symbols) |
| `name_shares_word` | at least one word in common |
| `name_indian_script` | match's name is written in an Indian script |
| `name_is_website` | match's name looks like `something.com` |
| `addr_empty` | match has no address |
| `addr_shares_word` | addresses share at least one word |
| `addr_shares_number` | addresses share at least one number (house / plot / door number) |

In [12]:
def share(list_a, list_b):
    return [len(set(a) & set(b)) > 0 for a, b in zip(list_a, list_b)]

s1_numbers = true_pairs["s1_business_address"].str.findall(r"\d+")
m_numbers = true_pairs["m_business_address"].str.findall(r"\d+")

checks = pd.DataFrame({
    "country": true_pairs["s1_country"],
    "source": true_pairs["m_source"],
    "name_exact": true_pairs["s1_business_name"] == true_pairs["m_business_name"],
    "name_same_lower": true_pairs["s1_business_name"].str.lower() == true_pairs["m_business_name"].str.lower(),
    "name_same_words": [set(a) == set(b) for a, b in zip(true_pairs["s1_name_words"], true_pairs["m_name_words"])],
    "name_shares_word": true_pairs["name_shares_word"],
    "name_indian_script": true_pairs["m_business_name"].str.contains("[\u0900-\u0D7F]"),
    "name_is_website": true_pairs["m_business_name"].str.contains(r"\.(?:com|net|org|in|co)\b", case=False),
    "addr_empty": true_pairs["m_business_address"] == "",
    "addr_shares_word": share(true_pairs["s1_addr_words"], true_pairs["m_addr_words"]),
    "addr_shares_number": share(s1_numbers, m_numbers),
})

evidence = (checks.groupby(["country", "source"]).mean() * 100).round(1).T
evidence["ALL"] = (checks.drop(columns=["country", "source"]).mean() * 100).round(1)
evidence

country            India          US         ALL
source                S2    S3    S2    S3      
name_exact           2.8   2.8   6.3   5.8   4.7
name_same_lower      6.3   6.7  14.6  13.0  10.8
name_same_words     20.5  22.3  32.9  32.1  28.0
name_shares_word    71.7  80.5  92.1  91.9  85.6
name_indian_script  23.0  12.8   0.0   0.0   7.2
name_is_website      4.6   4.5   5.5   5.5   5.1
addr_empty           3.8   4.2   5.2   4.7   4.6
addr_shares_word    96.2  95.8  94.8  95.3  95.4
addr_shares_number  83.5  81.4  76.1  80.1  79.9

**Reading this table**
- Only **10.8%** of true pairs have the same name after lowercasing. Exact name matching would miss ~90% of matches.
- **85.6%** share at least one name word; for India S2 only 71.7% (because 23% of India S2 names are in an Indian script).
- Addresses are the **most reliable** signal: 95.4% share a word, 79.9% share a number.
- ~4.6% of matches have an empty address — for those the name must carry the evidence.

### Name evidence vs address evidence together

Each true pair falls in one of four boxes:

In [13]:
name_ok = checks["name_shares_word"]
addr_ok = checks["addr_shares_number"]
box = pd.crosstab(
    name_ok.map({True: "name shares a word", False: "name shares NO word"}),
    addr_ok.map({True: "address shares a number", False: "address shares NO number"}),
    normalize="all",
) * 100
box.round(1)

addr_shares_number,address shares NO number,address shares a number
name_shares_word,,
name shares NO word,2.3,12.1
name shares a word,17.8,67.8


Only **2.3%** of true pairs share neither a name word nor an address number. So if blocking (Phase 7) looks at
**both** name words and address numbers/words, almost every true pair can be found.

In [14]:
print("True pairs with NO shared name word AND NO shared address number:")
true_pairs.loc[~name_ok & ~addr_ok, ["s1_business_name", "m_business_name", "s1_business_address", "m_business_address"]].sample(10, random_state=0)

True pairs with NO shared name word AND NO shared address number:


,s1_business_name,m_business_name,s1_business_address,m_business_address
40042,Loka (India) Holdings,lindia.com,"Co Bhutiya Rambhai Punjabhai, Limda Chowk Bhadrakali Road, Krishan...","H.NO 597 CO BHUTIYA RAMBHAI PUNJABHAI, LIMDA CHOWK BHADRAKALI ROAD..."
32218,North Food Private Limited,नॉर्थ फूड प्राइवेट लिमिटेड,"G-1205A Saviour Green Arch, Gh 10A, Techzone 4, Amrapali Dream Val...","उत्तर प्रदेश, BISHRAKH, G-01205A SAVIOUR GREEN ARCH, GAUTAM BUDDHA..."
15759,Best Sree Impex Private Limited,ಬೆಸ್ಟ್ ಶ್ರೀ ಇಂಪೆಕ್ಸ್ ಪ್ರೈವೇಟ್ ಲಿಮಿಟೆಡ್,"Bangalore, Unit No. F2117, 1St Floor, Block (B), Ardente Office On...","BANGALORE NORTH, UNIT NO. F211, BANGALORE, Karnataka"
58351,Real Projects,রিয়েল প্রজেক্টস,"C/O Benimadhab Roy, Chamtimukhi Laxmikanta Tea Garden, Jalpaiguri,...","C/O BENIMADHAB ROY, CHAMTIMUKHI LAXMIKANTA TEA GARDEN, JALPAIGURI,..."
18444,Shivam Exports Private Limited,शिवम एक्सपोर्ट्स प्राइवेट लिमिटेड,"Shop No. C-3, Gali No. 3, Dhawari Stadium, Dhawari, Satna, Madhya ...","SHOP NO. C-003, SATNA, Madhya Pradesh"
17739,Brittani Hoeppner Summit Acp LLC,brittanihoeppnersummitcom,"296 Old Lantern Road, Timberlake, NC","29 OLD LANTERN ROAD, TIMBERLAKE, NC"
23159,East Business Pvt Ltd,ईस्ट बिजनेस प्रा. लि.,"301 Surat Sada Surat Street Masjid East, Mumbai, Maharashtra","DOOR NO 3-01 SUROT SADA SURAT STREET MASJID EAST, MUMBAI, Maharashtra"
56902,"Hart, Roberts and Paige of Hennessey",#hartroberts,"525 Main Street, Hennessey, OK","Main St, HENNESSEY, OK"
52658,Royal Marketing Worldwide,royalmarketingworldwide.com,"38 Howlands Lane, Kingston, MA","038 HOWLANDS LN, KINGTSON, MA"
25672,Blue Hospitality Private Limited,બ્લૂ હોસ્પિટાલિટી પ્રાઇવેટ લિમિટેડ,"205 Vardhman Commercial, Office Opp S V School, Rajkot, Gujarat","VARDHMAN COMMERCIAL, OFFICE OPP S V SCHOOL, RAJKOT, Gujarat"


### Why does the name share no word? Break it down

In [15]:
no_word = checks[~checks["name_shares_word"]]
reasons = pd.Series({
    "match name in Indian script": no_word["name_indian_script"].mean() * 100,
    "match name is a website": no_word["name_is_website"].mean() * 100,
    "other (typos everywhere / different trade name)": (~no_word["name_indian_script"] & ~no_word["name_is_website"]).mean() * 100,
}).round(1)
reasons

match name in Indian script                        46.4
match name is a website                            32.7
other (typos everywhere / different trade name)    20.8
dtype: float64

When the names share no word, it's mostly because the match name is in an **Indian script** (46%) or is a
**website/hashtag** (33%, e.g. `owensmchenry.com`, `#pipefitterslocal`). Both are fixable later:
website names by comparing without spaces; Indian scripts by relying on the address (or transliteration).

---
## 4. Traps — same name, different business

A **false merge** (matching two different businesses) is what F0.5 punishes most.
How often does an S2/S3 record have **exactly the same lowercased name** (same country) as an S1 entity
but is **not** one of its true matches?

Steps:
1. Put the sampled S1 names in lowercase.
2. Scan Source 2 and 3 (name + country columns only, to save memory) and count records with each of those names.
3. Subtract the ones that are true matches — what is left are look-alikes (traps).

In [16]:
s1_keys = s1[["entity_id", "country"]].copy()
s1_keys["name_key"] = s1["business_name"].str.lower()

same_name_counts = []
for source_number in [2, 3]:
    df = pd.read_csv(TRAIN_DIR + f"train_source{source_number}.tsv", sep="\t", keep_default_na=False,
                     usecols=["entity_id", "business_name", "country"])
    df["name_key"] = df["business_name"].str.lower()
    df = df[df["name_key"].isin(s1_keys["name_key"])]
    same_name_counts.append(df[["entity_id", "name_key", "country"]])
    del df
    gc.collect()
same_name_records = pd.concat(same_name_counts, ignore_index=True)

# every (S1 entity, S2/S3 record) with the same lowercased name and country
same_name_pairs = s1_keys.merge(same_name_records, on=["name_key", "country"], suffixes=("_s1", "_other"))
true_pair_keys = set(zip(pairs["source1_entity_id"], pairs["match_id"]))
same_name_pairs["is_true_match"] = [
    (a, b) in true_pair_keys for a, b in zip(same_name_pairs["entity_id_s1"], same_name_pairs["entity_id_other"])
]
same_name_pairs["is_true_match"].value_counts()

is_true_match
False    108190
True       7513
Name: count, dtype: int64

In [17]:
traps_per_s1 = same_name_pairs[~same_name_pairs["is_true_match"]].groupby("entity_id_s1").size()
s1_keys["n_traps"] = s1_keys["entity_id"].map(traps_per_s1).fillna(0).astype(int)
s1_keys["is_singleton"] = s1_keys["entity_id"].isin(sample_gt.loc[sample_gt["n_true"] == 0, "source1_entity_id"])

summary = s1_keys.groupby(["country", "is_singleton"]).agg(
    entities=("entity_id", "size"),
    pct_with_a_trap=("n_traps", lambda x: round((x > 0).mean() * 100, 1)),
    max_traps=("n_traps", "max"),
)
summary

entities  pct_with_a_trap  max_traps
country is_singleton                                      
India   False             7611             35.6         50
        True               450             36.7         14
US      False            11259             30.5        320
        True               680             28.5        213

**This is the most important result of Phase 3.**

Among S2/S3 records with **exactly the same lowercased name** (same country) as a sampled S1 entity,
only ~7,500 are true matches and ~108,000 are **different businesses** — so "same name ⇒ match" would be right
only about **6.5%** of the time!

About a third of S1 entities (singletons included) have at least one same-name look-alike.
Generic names like `Family Center` have hundreds. Our model must **require address agreement**, not just name agreement.

In [18]:
trap_example = s1_keys.sort_values("n_traps", ascending=False).iloc[0]
print("S1 entity with the most look-alikes:", trap_example["entity_id"], "->", trap_example["n_traps"], "traps")
look_alike_ids = same_name_pairs.loc[same_name_pairs["entity_id_s1"] == trap_example["entity_id"], "entity_id_other"]

look_alikes = []
for source_number in [2, 3]:
    df = pd.read_csv(TRAIN_DIR + f"train_source{source_number}.tsv", sep="\t", keep_default_na=False)
    look_alikes.append(df[df["entity_id"].isin(look_alike_ids)])
    del df
    gc.collect()
look_alikes = pd.concat(look_alikes)
look_alikes["is_true_match"] = look_alikes["entity_id"].isin(pairs.loc[pairs["source1_entity_id"] == trap_example["entity_id"], "match_id"])

display(s1[s1["entity_id"] == trap_example["entity_id"]])
look_alikes.head(12)

S1 entity with the most look-alikes: S1-365712163 -> 320 traps


,entity_id,business_name,business_address,country
2597,S1-365712163,Family Center,"12605 8th Street, WA, Vancouver",US


,entity_id,business_name,business_address,country,is_true_match
3108,S2-698383825,Family Center,"194 4TH STREET, TROY, NY",US,False
17060,S2-151364780,FAMILY CENTER,"3502 FOURTEENTH STREET, WASHINGTON, DC",US,False
52651,S2-505625136,FAMILY CENTER,"#157 BARCELONA DRIVE, WILTON, NY",US,False
75596,S2-785200183,Family Center,"2053 WILLOW BROOKE DR, WOODSTOCK, IL",US,False
95273,S2-114212739,Family Center,"11272 JTHN F KENNEDY DRIVE, HAGERSTOWN, MD",US,False
136336,S2-445147077,Family Center,"MAYOWOD, IL, 2018 Seventh Ave",US,False
204950,S2-255041975,Family Center,"9 SHORT ST, LAWRENCE, MA",US,False
225591,S2-940528074,Family Center,"TACOMA CTY, WA, 4702 87TH AVENUE",US,False
261810,S2-322308888,Family Center,"Dewey Avenue, HERKIMER, NY",US,False
306055,S2-249453502,Family Center,"#667 TRAILSIDE ROAD, GENESEE, CO",US,False


---
# Summary — Phase 3

| What we measured | Result | What it means for us |
|---|---|---|
| Score for "predict nothing" | 0.056 | the floor to beat |
| True pairs with the same lowercased name | 10.8% | exact matching is useless on its own |
| True pairs sharing ≥1 name word | 85.6% | name words are useful for blocking, not sufficient |
| True pairs sharing ≥1 address word / number | 95.4% / 79.9% | the address is the strongest evidence |
| True pairs sharing neither name word nor address number | 2.3% | combined blocking can reach high recall |
| "Same name" pairs that are actually different businesses | ~93.5% | name alone causes false merges → need address agreement |
| S1 entities with at least one same-name look-alike | ~30–37% | hard negatives are everywhere, including singletons |

Also: trade names (`née ...`), completely different names linked by address, websites/hashtags as names,
names in 9 Indian scripts, and fake numbers injected into addresses.
